# 危机分级可行性实验（P6 前置）

> **说明**：本 notebook 只做研究性的可行性评估，不构成任何诊断，产出的模型**不进入产品**。

## 要回答的问题

规则模式对不同作者写的句子识别能力很低（52 句探针上 L3 0/12）。这个实验回答：
**一个现成的中文预训练模型，配上公开数据和项目自己的用例，能不能在召回明显超过规则的同时，把日常说法的误报压在可接受范围内？**

## 第一轮结果与本版改动（2026-10-22）

第一轮（单次划分、验证集 57 个正例上定阈值）在探针上 L2+L3 召回 32/32，但 L0 误报 10/14。
模型本身区分得开（探针上 AUC 0.99），问题出在阈值：在很小的验证集上要求 98% 召回，又遇上过度自信的模型，阈值被压到 0.014。

本版改为：

1. **5 折交叉验证**：项目用例 + SOS-HL-1K 全部约 1,900 条都得到一次“训练时没见过”的预测（OOF），阈值在这上面定，正例数从 57 增加到约 380。
2. **温度缩放**：在 OOF 预测上为两个头各拟合一个温度，修正过度自信，再定阈值。
3. **固定训练轮数**：各折与最终模型都训练 `EPOCHS` 轮，不再按验证集早停（早停会让验证集参与选模型）。
4. **测试集分开报告**：52 句探针的模型结果第一轮已经看过，**不再是独立测试**，只作参考；真正的结论要看独立评估集（`role=fresh`，由不同作者编写，导出时用 `--fresh-eval` 加入）。

## 数据

| 文件（放在 `DRIVE_ROOT/raw/` 下） | 来源 | 用途 |
| :--- | :--- | :--- |
| `olivia/olivia_cases.csv` | 本地运行 `training/colab/export_olivia_data.py` 生成后上传 | `role=train`：交叉验证池；`role=probe`：52 句探针（已看过）；`role=fresh`：独立评估集（可选） |
| `sos_hl_1k/suicide_train_BERT.tsv`、`suicide_val_BERT.tsv` | SOS-HL-1K（微博“走饭”评论，心理学人员标注高 / 低自杀风险）。不存在时按固定提交号自动下载并校验 SHA-256 | 高风险头，两份文件都进交叉验证池 |

SOS-HL-1K 的“高风险”大致对应项目的 L3，“低风险”里混有 L0–L2，所以它只训练二分类的高风险头。该仓库没有声明许可证：只用于本实验，权重不得随产品分发。

## 运行

1. 本地仓库根目录运行 `.venv/bin/python training/colab/export_olivia_data.py`（有独立评估集时加 `--fresh-eval 文件.csv`），把 `build/colab_data/olivia_cases.csv` 上传到 Drive 的 `OLIVIA_crisis/raw/olivia/`（覆盖旧文件）。
2. Colab：运行时类型选 GPU，“运行时 → 全部运行”。T4 上约 25 分钟（5 折 + 最终模型）。
3. 断开后重新“全部运行”是安全的：已完成的折和最终模型在配置与数据未变时直接复用；每个测试集只测一次（除非 `FORCE_RETEST = True`）。

## 产出（`DRIVE_ROOT/` 下）

| 路径 | 内容 |
| :--- | :--- |
| `models/crisis_teacher/model.pt`、`train_config.json` | 最终模型（全部训练数据，`EPOCHS` 轮）与配置 |
| `models/cv_folds/fold_*.npz` | 各折对留出部分的预测（原始 logits） |
| `reports/oof_predictions.csv` | 每条训练数据的 OOF 校准后概率 |
| `configs/calibration.json` | 温度、阈值与 OOF 指标（含按类别的误报率，`HYPERBOLE` 一行是 G-2 口癖误报的粗略估计） |
| `reports/{probe,fresh}_predictions.csv`、`_summary.md`、`_summary.json` | 各测试集逐条结果与汇总 |

## 不要做的事

- 不要按探针或独立评估集的结果回头改阈值、温度或超参再测。
- OOF 指标来自与训练数据同一作者的句子，会偏乐观；它只用来定阈值，结论以独立评估集为准。

## 1. 环境

In [ ]:
# 依赖：直接用 Colab 预装的 torch / transformers / scikit-learn / pandas
# （Colab 已是 Python 3.13，固定旧版本会触发源码编译，既慢又可能失败；实际版本在下一单元打印）
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
# 环境检查、随机种子与挂载 Drive
import csv, hashlib, html, json, math, os, random, re, time, urllib.request

import numpy as np
import pandas as pd
import torch
import transformers
from torch import nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | transformers {transformers.__version__} | device {DEVICE}")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
elif IN_COLAB:
    print("提示：当前没有 GPU。请在“运行时 → 更改运行时类型”中选择 GPU 后重新运行。")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

## 2. 配置区（所有可变项集中于此）

In [ ]:
SMOKE_TEST = os.environ.get("OLIVIA_SMOKE_TEST") == "1"  # 本地 CPU 冒烟测试用；Colab 上保持关闭
DRIVE_ROOT = os.environ.get("OLIVIA_CRISIS_ROOT", "/content/drive/MyDrive/OLIVIA_crisis")

ENCODER = "hfl/chinese-roberta-wwm-ext"
SEED = 42
MAX_LEN = 160         # 含 [CLS]/[SEP] 的 token 上限
HEAD_TOKENS = 48      # 超长时保留开头这么多 token，其余从结尾取（与 SDD §10.1 双端保全同理）
BATCH_SIZE = 32
EPOCHS = 4            # 固定轮数：各折与最终模型相同（第一轮验证指标在第 4–5 轮见顶，之后训练损失趋近 0）
LR = 3e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
HIGH_LOSS_WEIGHT = 1.0
CV_FOLDS = 5
OURS_FEED_HIGH_HEAD = True  # 项目用例也训练高风险头：L3 记为高，L0/L1 记为低，L2 不参与

TARGET_RECALL_GE2 = 0.98   # OOF 上 L2+L3 的召回目标（G-2 危机召回 ≥ 98%）
TARGET_RECALL_HIGH = 0.95  # OOF 上 SOS 高风险的召回目标
FP_CATEGORIES = ("HYPERBOLE", "BENIGN", "NEGATION_PURE", "NEGATION_WITH_MEANS")  # 单独报告误报率的用例类别

FORCE_RETRAIN = False
FORCE_RETEST = False

SOS_COMMIT = "78fb4d1ebfa0addb58417c7ad6e36c36650181c3"
SOS_URL = "https://raw.githubusercontent.com/HongzhiQ/SupervisedVsLLM-EfficacyEval/{commit}/data/suicide/{name}"
SOS_ENCODING = "gb18030"  # 原文件是 GBK 编码、CRLF 换行（GB18030 是 GBK 的超集）
SOS_FILES = {
    "train": ("suicide_train_BERT.tsv", "443b84b944d99ab357f3e6a584bd137c642d6e8557f89b8ef76ac8abe4727cc2"),
    "val": ("suicide_val_BERT.tsv", "15f76aae581a678adea3f43e1f0c96d6f5e79b782e0c113c440f12213343e2a1"),
}
PROBE_SIZE = 52
LEVEL_NAMES = ["L0", "L1", "L2", "L3"]
TEST_SETS = {
    "probe": "52 句探针（模型结果已于 2026-10-22 看过，不再是独立测试，仅作参考）",
    "fresh": "独立评估集（不同作者编写）",
}

PATHS = {
    "olivia_csv": os.path.join(DRIVE_ROOT, "raw", "olivia", "olivia_cases.csv"),
    "sos_dir": os.path.join(DRIVE_ROOT, "raw", "sos_hl_1k"),
    "model_dir": os.path.join(DRIVE_ROOT, "models", "crisis_teacher"),
    "fold_dir": os.path.join(DRIVE_ROOT, "models", "cv_folds"),
    "reports": os.path.join(DRIVE_ROOT, "reports"),
    "configs": os.path.join(DRIVE_ROOT, "configs"),
}
PATHS["model"] = os.path.join(PATHS["model_dir"], "model.pt")
PATHS["train_config"] = os.path.join(PATHS["model_dir"], "train_config.json")
PATHS["calibration"] = os.path.join(PATHS["configs"], "calibration.json")
PATHS["oof"] = os.path.join(PATHS["reports"], "oof_predictions.csv")

if SMOKE_TEST:
    EPOCHS, BATCH_SIZE, MAX_LEN, HEAD_TOKENS, CV_FOLDS = 1, 8, 64, 16, 2

for key in ("sos_dir", "model_dir", "fold_dir", "reports", "configs"):
    os.makedirs(PATHS[key], exist_ok=True)
set_seed(SEED)
print("DRIVE_ROOT:", DRIVE_ROOT, "| SMOKE_TEST:", SMOKE_TEST)

## 3. 数据：下载校验、读取、分折与泄露检查

In [ ]:
def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def fetch_sos():
    # 缺失时按固定提交号下载；已存在也校验 SHA-256，内容不符直接报错
    paths = {}
    for split, (name, expected) in SOS_FILES.items():
        path = os.path.join(PATHS["sos_dir"], name)
        if not os.path.exists(path):
            url = SOS_URL.format(commit=SOS_COMMIT, name=name)
            print("下载", url)
            urllib.request.urlretrieve(url, path + ".part")
            os.replace(path + ".part", path)
        actual = sha256_of(path)
        if actual != expected:
            raise ValueError(f"{name} 的 SHA-256 不符：{actual}（应为 {expected}）")
        paths[split] = path
    return paths


def clean_text(text):
    text = html.unescape(str(text)).replace(" ", " ")
    return re.sub(r"\s+", " ", text).strip()


def load_sos(path, split):
    # 三列无表头：label（1=高风险）、其反码、正文
    df = pd.read_csv(path, sep="\t", header=None, encoding=SOS_ENCODING, names=["label", "label_inverse", "text"],
                     dtype={"text": str}, quoting=csv.QUOTE_NONE, keep_default_na=False)
    if not ((df["label"] + df["label_inverse"]) == 1).all():
        raise ValueError(f"{path}: 第 1、2 列不是互补的 0/1 标签")
    return pd.DataFrame({"id": [f"sos_{split}_{i}" for i in range(len(df))],
                         "text": df["text"].map(clean_text), "level": -1,
                         "high": df["label"].astype(int), "source": "sos_hl_1k",
                         "category": "", "rule_level": -1})


def load_olivia(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"找不到 {path}：先在本地运行 export_olivia_data.py 并上传到这里")
    df = pd.read_csv(path, dtype={"id": str, "text": str, "role": str, "source": str, "category": str},
                     keep_default_na=False)
    missing = {"id", "text", "level", "role", "source", "rule_level"} - set(df.columns)
    if missing:
        raise ValueError(f"olivia_cases.csv 缺少列：{sorted(missing)}（请用最新的 export_olivia_data.py 重新导出）")
    if "category" not in df.columns:
        df["category"] = ""
    if not df["level"].isin([0, 1, 2, 3]).all() or not df["role"].isin(["train", *TEST_SETS]).all():
        raise ValueError("olivia_cases.csv 的 level 或 role 取值非法")
    if (df["role"] == "probe").sum() != PROBE_SIZE:
        raise ValueError(f"探针应为 {PROBE_SIZE} 条，实际 {(df['role'] == 'probe').sum()} 条")
    df["text"] = df["text"].map(clean_text)
    df["high"] = df["level"].map({0: 0, 1: 0, 2: -1, 3: 1}) if OURS_FEED_HIGH_HEAD else -1
    return df


def assign_folds(frame, folds, seed):
    # 按 来源 × 等级 × 高风险 分层，每层内打乱后轮流分到各折
    rng = np.random.default_rng(seed)
    fold = np.empty(len(frame), dtype=int)
    for _, idx in frame.groupby(["source", "level", "high"]).indices.items():
        idx = rng.permutation(idx)
        fold[idx] = np.arange(len(idx)) % folds
    return fold


def bigrams(text):
    t = re.sub(r"\s+", "", text)
    return {t[i:i + 2] for i in range(len(t) - 1)} or {t}


def drop_test_leaks(frame, test_texts, threshold=0.8):
    # 训练池中与任何测试句相同或高度相似（字二元组 Jaccard ≥ 0.8）的句子一律剔除
    test_grams = [bigrams(t) for t in test_texts]
    leak = frame["text"].map(
        lambda t: any(len(bigrams(t) & g) / len(bigrams(t) | g) >= threshold for g in test_grams))
    for _, row in frame[leak].iterrows():
        print(f"  剔除疑似泄露 [{row['source']}] {row['text'][:60]}")
    return frame[~leak]


sos_paths = fetch_sos()
olivia = load_olivia(PATHS["olivia_csv"])
tests = {role: olivia[olivia["role"] == role].reset_index(drop=True) for role in TEST_SETS}
pool = pd.concat([olivia[olivia["role"] == "train"], load_sos(sos_paths["train"], "train"),
                  load_sos(sos_paths["val"], "val")], ignore_index=True)
if SMOKE_TEST:
    pool = pd.concat([g.sample(min(len(g), 48), random_state=SEED) for _, g in pool.groupby("source")])

print("泄露检查：")
pool = drop_test_leaks(pool, pd.concat([t["text"] for t in tests.values()])).reset_index(drop=True)
pool["fold"] = assign_folds(pool, CV_FOLDS, SEED)

print("\n交叉验证池：", pool.groupby(["source", "level", "high"]).size().to_dict())
print("每折条数：", pool["fold"].value_counts().sort_index().to_dict())
for role, frame in tests.items():
    counts = frame["level"].map(dict(enumerate(LEVEL_NAMES))).value_counts().sort_index().to_dict()
    print(f"测试集 {role}：{len(frame)} 条 {counts}")
DATA_FINGERPRINT = hashlib.sha256(pool[["text", "level", "high", "fold"]].to_csv(index=False).encode("utf-8")).hexdigest()[:16]

## 4. 模型：共享编码器 + 四级头 + 高风险头

In [ ]:
from transformers import AutoModel, AutoTokenizer, BertConfig, BertModel, get_linear_schedule_with_warmup

tokenizer = AutoTokenizer.from_pretrained(ENCODER)


class CrisisModel(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        hidden = encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.level_head = nn.Linear(hidden, len(LEVEL_NAMES))
        self.high_head = nn.Linear(hidden, 1)

    def forward(self, input_ids, attention_mask):
        cls = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        cls = self.dropout(cls)
        return self.level_head(cls), self.high_head(cls).squeeze(-1)


def build_model():
    if SMOKE_TEST:  # 冒烟测试只验证流程：随机初始化的极小编码器，不下载预训练权重
        cfg = BertConfig(vocab_size=tokenizer.vocab_size, hidden_size=32, num_hidden_layers=1,
                         num_attention_heads=2, intermediate_size=64, max_position_embeddings=MAX_LEN + 2)
        return CrisisModel(BertModel(cfg))
    return CrisisModel(AutoModel.from_pretrained(ENCODER))


def encode(text):
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    budget = MAX_LEN - 2
    if len(ids) > budget:
        ids = ids[:HEAD_TOKENS] + ids[len(ids) - (budget - HEAD_TOKENS):]
    return [tokenizer.cls_token_id] + ids + [tokenizer.sep_token_id]


def batches(frame, batch_size, shuffle=False, seed=SEED):
    order = np.arange(len(frame))
    if shuffle:
        np.random.default_rng(seed).shuffle(order)
    for start in range(0, len(order), batch_size):
        rows = frame.iloc[order[start:start + batch_size]]
        encoded = [encode(t) for t in rows["text"]]
        width = max(len(e) for e in encoded)
        ids = torch.full((len(encoded), width), tokenizer.pad_token_id, dtype=torch.long)
        mask = torch.zeros((len(encoded), width), dtype=torch.long)
        for i, e in enumerate(encoded):
            ids[i, :len(e)] = torch.tensor(e)
            mask[i, :len(e)] = 1
        yield (ids.to(DEVICE), mask.to(DEVICE),
               torch.tensor(rows["level"].to_numpy(), dtype=torch.long, device=DEVICE),
               torch.tensor(rows["high"].to_numpy(), dtype=torch.float, device=DEVICE))

## 5. 训练：5 折交叉验证 + 全量最终模型（已完成且配置未变时复用）

In [ ]:
AMP_DTYPE = None
if DEVICE.type == "cuda":
    AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

TRAIN_CONFIG = {
    "encoder": "smoke-tiny-random" if SMOKE_TEST else ENCODER, "seed": SEED, "max_len": MAX_LEN,
    "head_tokens": HEAD_TOKENS, "batch_size": BATCH_SIZE, "epochs": EPOCHS, "lr": LR,
    "weight_decay": WEIGHT_DECAY, "warmup_ratio": WARMUP_RATIO, "high_loss_weight": HIGH_LOSS_WEIGHT,
    "cv_folds": CV_FOLDS, "ours_feed_high_head": OURS_FEED_HIGH_HEAD, "sos_commit": SOS_COMMIT,
    "data_fingerprint": DATA_FINGERPRINT,
}
CONFIG_FINGERPRINT = hashlib.sha256(json.dumps(TRAIN_CONFIG, sort_keys=True).encode()).hexdigest()[:16]


def loss_fns(frame):
    levels = frame.loc[frame["level"] >= 0, "level"]
    counts = np.bincount(levels, minlength=len(LEVEL_NAMES)).astype(float)
    weights = np.sqrt(counts.sum() / (len(LEVEL_NAMES) * np.maximum(counts, 1)))
    highs = frame.loc[frame["high"] >= 0, "high"]
    pos_weight = (highs == 0).sum() / max((highs == 1).sum(), 1)
    ce = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float, device=DEVICE))
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, dtype=torch.float, device=DEVICE))

    def loss(level_logits, high_logits, level, high):
        total = level_logits.new_zeros(())
        has_level, has_high = level >= 0, high >= 0
        if has_level.any():
            total = total + ce(level_logits[has_level].float(), level[has_level])
        if has_high.any():
            total = total + HIGH_LOSS_WEIGHT * bce(high_logits[has_high].float(), high[has_high])
        return total
    return loss


def train_model(frame, seed, tag):
    set_seed(seed)
    model = build_model().to(DEVICE)
    loss = loss_fns(frame)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps = EPOCHS * math.ceil(len(frame) / BATCH_SIZE)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)
    for epoch in range(1, EPOCHS + 1):
        model.train()
        started, total = time.time(), 0.0
        for ids, mask, level, high in batches(frame, BATCH_SIZE, shuffle=True, seed=seed + epoch):
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
                level_logits, high_logits = model(ids, mask)
            batch_loss = loss(level_logits, high_logits, level, high)
            optimizer.zero_grad()
            scaler.scale(batch_loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total += batch_loss.item()
        print(f"  [{tag}] epoch {epoch}/{EPOCHS} train_loss {total:.3f} ({time.time() - started:.0f}s)")
    return model


@torch.no_grad()
def predict_logits(model, frame):
    model.eval()
    level, high = [], []
    for ids, mask, _, _ in batches(frame, BATCH_SIZE * 2):
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            level_logits, high_logits = model(ids, mask)
        level.append(level_logits.float().cpu().numpy())
        high.append(high_logits.float().cpu().numpy())
    return np.concatenate(level), np.concatenate(high)


def release(model):
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


# ---- 交叉验证：每折训练一个模型，只预测留出的那一折 ----
oof_level = np.full((len(pool), len(LEVEL_NAMES)), np.nan, dtype=np.float32)
oof_high = np.full(len(pool), np.nan, dtype=np.float32)
for k in range(CV_FOLDS):
    fold_path = os.path.join(PATHS["fold_dir"], f"fold_{k}.npz")
    held = np.flatnonzero(pool["fold"].to_numpy() == k)
    if os.path.exists(fold_path) and not FORCE_RETRAIN:
        saved = np.load(fold_path)
        if str(saved["fingerprint"]) == CONFIG_FINGERPRINT and np.array_equal(saved["index"], held):
            oof_level[held], oof_high[held] = saved["level"], saved["high"]
            print(f"第 {k} 折已完成，复用。")
            continue
    print(f"第 {k} 折：训练 {len(pool) - len(held)} 条，预测 {len(held)} 条")
    model = train_model(pool.drop(index=held), SEED + 100 * (k + 1), f"fold {k}")
    level_logits, high_logits = predict_logits(model, pool.iloc[held])
    np.savez(fold_path, fingerprint=CONFIG_FINGERPRINT, index=held, level=level_logits, high=high_logits)
    oof_level[held], oof_high[held] = level_logits, high_logits
    release(model)
assert not np.isnan(oof_level).any() and not np.isnan(oof_high).any()


# ---- 最终模型：全部训练数据，同样的轮数 ----
def final_model_is_current():
    if FORCE_RETRAIN or not (os.path.exists(PATHS["model"]) and os.path.exists(PATHS["train_config"])):
        return False
    with open(PATHS["train_config"], encoding="utf-8") as f:
        saved = json.load(f)
    return saved.get("training_complete") and saved.get("config_fingerprint") == CONFIG_FINGERPRINT


if final_model_is_current():
    print("最终模型已训练完成且配置一致，复用。")
else:
    with open(PATHS["train_config"], "w", encoding="utf-8") as f:
        json.dump({**TRAIN_CONFIG, "config_fingerprint": CONFIG_FINGERPRINT, "training_complete": False}, f, indent=2)
    model = train_model(pool, SEED, "final")
    torch.save({"state_dict": model.state_dict(), "config": TRAIN_CONFIG}, PATHS["model"])
    release(model)
    with open(PATHS["train_config"], "w", encoding="utf-8") as f:
        json.dump({**TRAIN_CONFIG, "config_fingerprint": CONFIG_FINGERPRINT, "training_complete": True}, f, indent=2)

model = build_model().to(DEVICE)
model.load_state_dict(torch.load(PATHS["model"], map_location=DEVICE)["state_dict"])
model.eval()
print("已加载最终模型，配置指纹", CONFIG_FINGERPRINT)

## 6. 校准：温度缩放 + 在 OOF 预测上定阈值

温度只改变概率的“自信程度”，不改变排序；阈值在校准后的 OOF 概率上按召回目标确定。
最终模型用全部数据训练，它的校准与各折模型相近但不完全相同，这是交叉验证定阈值的通常近似。

In [ ]:
from sklearn.metrics import roc_auc_score


LOG_T_RANGE = (math.log(0.05), math.log(20.0))  # 温度限制在 [0.05, 20]，防止无信息的 logits 把温度推向无穷


def fit_temperature(logits, labels, binary):
    # 在 log 温度上用 LBFGS 最小化负对数似然
    x = torch.tensor(logits, dtype=torch.float64)
    y = torch.tensor(labels, dtype=torch.float64 if binary else torch.long)
    log_t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    nll = nn.BCEWithLogitsLoss() if binary else nn.CrossEntropyLoss()
    optimizer = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200)

    def closure():
        optimizer.zero_grad()
        value = nll(x / log_t.clamp(*LOG_T_RANGE).exp(), y)
        value.backward()
        return value
    optimizer.step(closure)
    with torch.no_grad():
        t = log_t.clamp(*LOG_T_RANGE).exp()
        before, after = nll(x, y).item(), nll(x / t, y).item()
    return float(t), before, after


def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def threshold_for_recall(scores, positives, target):
    # 满足“召回 ≥ target”的最大阈值：允许漏掉 floor((1-target)*正例数) 个正例
    pos = np.sort(scores[positives])
    if len(pos) == 0:
        return 0.5
    allowed_misses = int(math.floor((1 - target) * len(pos) + 1e-9))
    return float(pos[allowed_misses])


def model_levels(p_level, p_high, cal):
    level = p_level.argmax(1)
    level = np.where(p_level[:, 2] + p_level[:, 3] >= cal["threshold_ge2"], np.maximum(level, 2), level)
    return np.where(p_high >= cal["threshold_high"], 3, level)


level_rows = pool["level"].to_numpy() >= 0
high_rows = pool["high"].to_numpy() >= 0
sos_rows = (pool["source"] == "sos_hl_1k").to_numpy()
t_level, nll_level_before, nll_level_after = fit_temperature(oof_level[level_rows], pool["level"].to_numpy()[level_rows], False)
t_high, nll_high_before, nll_high_after = fit_temperature(oof_high[high_rows], pool["high"].to_numpy()[high_rows].astype(float), True)
p_level_oof = softmax(oof_level / t_level)
p_high_oof = sigmoid(oof_high / t_high)
p_ge2_oof = p_level_oof[:, 2] + p_level_oof[:, 3]

y_level = pool["level"].to_numpy()
y_high = pool["high"].to_numpy()
CALIBRATION = {
    "config_fingerprint": CONFIG_FINGERPRINT,
    "temperature_level": t_level, "temperature_high": t_high,
    "nll_level": [nll_level_before, nll_level_after], "nll_high": [nll_high_before, nll_high_after],
    "threshold_ge2": threshold_for_recall(p_ge2_oof[level_rows], y_level[level_rows] >= 2, TARGET_RECALL_GE2),
    "threshold_high": threshold_for_recall(p_high_oof[sos_rows], y_high[sos_rows] == 1, TARGET_RECALL_HIGH),
}

lv = model_levels(p_level_oof, p_high_oof, CALIBRATION)
ours = level_rows


def rate(mask, cond):
    n = int(mask.sum())
    return {"hit": int((mask & cond).sum()), "n": n, "rate": float((mask & cond).sum() / n) if n else float("nan")}


category = pool["category"].to_numpy()
CALIBRATION["oof"] = {
    "auc_ge2_vs_rest": float(roc_auc_score(y_level[ours] >= 2, p_ge2_oof[ours])),
    "auc_high_sos": float(roc_auc_score(y_high[sos_rows], p_high_oof[sos_rows])),
    "L2+L3 判为 ≥L2": rate(ours & (y_level >= 2), lv >= 2),
    "L3 判为 L3（模型）": rate(ours & (y_level == 3), lv == 3),
    "L1 判为 L1": rate(ours & (y_level == 1), lv == 1),
    "L0 误报为 ≥L2": rate(ours & (y_level == 0), lv >= 2),
    "SOS 低风险误判为高风险": rate(sos_rows & (y_high == 0), p_high_oof >= CALIBRATION["threshold_high"]),
    "按类别误报为 ≥L2": {c: rate(ours & (category == c), lv >= 2) for c in FP_CATEGORIES},
}
with open(PATHS["calibration"], "w", encoding="utf-8") as f:
    json.dump(CALIBRATION, f, ensure_ascii=False, indent=2)
pool.assign(p_L0=p_level_oof[:, 0], p_L1=p_level_oof[:, 1], p_L2=p_level_oof[:, 2], p_L3=p_level_oof[:, 3],
            p_high=p_high_oof, model_level=lv).drop(columns=["text"]).to_csv(PATHS["oof"], index=False)

print(f"温度：四级头 {t_level:.2f}（NLL {nll_level_before:.3f} → {nll_level_after:.3f}），"
      f"高风险头 {t_high:.2f}（NLL {nll_high_before:.3f} → {nll_high_after:.3f}）")
print(f"阈值：ge2 {CALIBRATION['threshold_ge2']:.4f}，high {CALIBRATION['threshold_high']:.4f}")
print(json.dumps(CALIBRATION["oof"], ensure_ascii=False, indent=1))

## 7. 测试集评估（每个测试集只运行一次）

模型等级：四级头取最大概率；`P(L2)+P(L3)` 过 ge2 阈值时至少 L2；高风险头过 high 阈值时记为 L3。
合并等级按 `crisis.py` 现行仲裁：规则 L3（Stage 1 命中）保持 L3；否则取规则与模型的较高者，最高 L2。

In [ ]:
def combined_levels(rule, model_level):
    return np.where(rule == 3, 3, np.minimum(np.maximum(rule, model_level), 2))


def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return (max(0.0, centre - half), min(1.0, centre + half))


def summarize(ideal, predicted):
    def count(mask, cond):
        return int((mask & cond).sum()), int(mask.sum())
    rows = {
        "L3 判为 L3": count(ideal == 3, predicted == 3),
        "L3 判为 ≥L2（弹热线卡）": count(ideal == 3, predicted >= 2),
        "L2 判为 ≥L2": count(ideal == 2, predicted >= 2),
        "L2+L3 判为 ≥L2": count(ideal >= 2, predicted >= 2),
        "L1 判为 L1": count(ideal == 1, predicted == 1),
        "L0 误报为 ≥L2": count(ideal == 0, predicted >= 2),
        "L0 误报为 ≥L1": count(ideal == 0, predicted >= 1),
    }
    return {name: {"hit": k, "n": n, "ci95": wilson(k, n)} for name, (k, n) in rows.items()}


def evaluate(role, frame):
    level_logits, high_logits = predict_logits(model, frame)
    p_level = softmax(level_logits / CALIBRATION["temperature_level"])
    p_high = sigmoid(high_logits / CALIBRATION["temperature_high"])
    ideal, rule = frame["level"].to_numpy(), frame["rule_level"].to_numpy()
    model_level = model_levels(p_level, p_high, CALIBRATION)
    combined = combined_levels(rule, model_level)
    predictions = frame[["id", "text", "level", "rule_level"]].rename(columns={"level": "ideal_level"}).assign(
        **{f"p_{name}": p_level[:, i].round(4) for i, name in enumerate(LEVEL_NAMES)},
        p_high=p_high.round(4), model_level=model_level, combined_level=combined)
    predictions.to_csv(os.path.join(PATHS["reports"], f"{role}_predictions.csv"), index=False, encoding="utf-8")
    summary = {"test_set": TEST_SETS[role], "config_fingerprint": CONFIG_FINGERPRINT,
               "calibration": {k: v for k, v in CALIBRATION.items() if k != "oof"}, "variants": {
                   "规则（现状）": summarize(ideal, rule),
                   "模型单独（未经仲裁）": summarize(ideal, model_level),
                   "规则 + 模型（按现行仲裁）": summarize(ideal, combined)}}
    with open(os.path.join(PATHS["reports"], f"{role}_summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    names = list(next(iter(summary["variants"].values())))
    lines = [f"# 危机分级可行性实验：{TEST_SETS[role]}", "",
             f"{len(frame)} 条。配置指纹 `{CONFIG_FINGERPRINT}`；温度与阈值只用交叉验证的 OOF 预测确定。区间为 Wilson 95% 置信区间。", "",
             "| 指标 | " + " | ".join(summary["variants"]) + " |",
             "| :--- | " + " | ".join(":---:" for _ in summary["variants"]) + " |"]
    for name in names:
        cells = [f"{v[name]['hit']}/{v[name]['n']}（{v[name]['ci95'][0]:.0%}–{v[name]['ci95'][1]:.0%}）"
                 for v in summary["variants"].values()]
        lines.append(f"| {name} | " + " | ".join(cells) + " |")
    sections = [
        ("规则漏判、规则 + 模型补上的句子", lambda r: r.ideal_level >= 2 and r.rule_level < 2 and r.combined_level >= 2),
        ("规则 + 模型仍然漏判（应为 ≥L2，合并 < L2）", lambda r: r.ideal_level >= 2 and r.combined_level < 2),
        ("新增误报（应为 L0，规则 < L2，合并 ≥ L2）", lambda r: r.ideal_level == 0 and r.rule_level < 2 and r.combined_level >= 2),
    ]
    for title, keep in sections:
        lines += ["", f"## {title}", ""]
        lines += [f"- {r.id}（应为 {LEVEL_NAMES[r.ideal_level]}，规则 {LEVEL_NAMES[r.rule_level]}，合并 {LEVEL_NAMES[r.combined_level]}，"
                  f"P(L2)+P(L3)={r.p_L2 + r.p_L3:.2f}，p_high={r.p_high:.2f}）{r.text}"
                  for r in predictions.itertuples() if keep(r)]
    with open(os.path.join(PATHS["reports"], f"{role}_summary.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")


fingerprint = f"{CONFIG_FINGERPRINT}:{CALIBRATION['threshold_ge2']:.6f}:{CALIBRATION['threshold_high']:.6f}"
for role, frame in tests.items():
    if frame.empty:
        print(f"测试集 {role} 为空，跳过（独立评估集需要在导出时用 --fresh-eval 加入）。")
        continue
    marker = os.path.join(PATHS["reports"], f".evaluated_{role}")
    if os.path.exists(marker) and not FORCE_RETEST:
        with open(marker, encoding="utf-8") as f:
            print(f"测试集 {role} 已测过（{f.read().strip()}），不重复测试；下面显示上次的结果。")
    else:
        evaluate(role, frame)
        with open(marker, "w", encoding="utf-8") as f:
            f.write(f"{fingerprint} @ {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    with open(os.path.join(PATHS["reports"], f"{role}_summary.md"), encoding="utf-8") as f:
        print(f.read())

## 8. 怎么读结果

- 先看第 6 节的 OOF 指标：`L2+L3 判为 ≥L2` 应接近召回目标；`L0 误报` 与 `HYPERBOLE` 类别的误报率是代价。它们来自同一作者的句子，偏乐观。
- 再看独立评估集（`fresh`）：这是唯一能下结论的数字。主要看 **“L2+L3 判为 ≥L2”** 和 **“L0 误报为 ≥L2”** 两行要一起看。
- 52 句探针（`probe`）的模型结果第一轮已经看过，只作参考，不能用来证明改进。
- “L3 判为 L3” 在合并一栏不会因模型而提高：没有 Stage 1 计划证据时，现行仲裁最高给 L2（防止误报 110/120）。

In [ ]:
# 各产出文件是否存在及大小
outputs = [PATHS["model"], PATHS["train_config"], PATHS["calibration"], PATHS["oof"]]
outputs += [os.path.join(PATHS["fold_dir"], f"fold_{k}.npz") for k in range(CV_FOLDS)]
for role in TEST_SETS:
    outputs += [os.path.join(PATHS["reports"], f"{role}_{suffix}") for suffix in ("predictions.csv", "summary.md", "summary.json")]
for path in outputs:
    size = f"{os.path.getsize(path) / 1024:.1f} KB" if os.path.exists(path) else "缺失"
    print(f"{size:>12s}  {path}")